# Verify a robot controller

Verifies a robot controller trained by Train_RCCO. The exported bundle is loaded through `GraphRobotController`, the same path used to drive the robot, and fed every recorded frame of the testing demonstrations (teacher forcing). The predicted normalized actions are compared to the recorded actions at the next timestep.

The controller is specified by the training recipe exp/run named in an experiment of type robot_controller_verify. See robot_controller/DESIGN-BehaviorCloningFlow.md.

In [ ]:
import sys
sys.path.append("..")
from exp_run_config import Config
Config.PROJECTNAME = "BerryPicker"

import pathlib
from pprint import pprint
import matplotlib.pyplot as plt
# fixing the fact that it defaults to Type 3 fonts
import matplotlib
matplotlib.rcParams['pdf.fonttype'] = 42
matplotlib.rcParams['ps.fonttype'] = 42
import numpy as np

from robot.al5d.position import RobotPosition
from robot_controller.controller_verification import teacher_forcing
from robot_controller.roco_factory import create_controller

In [ ]:
# *** Initialize the variables with default values
# *** This cell is tagged as parameters
# *** If papermill is used, some of the values will be overwritten

creation_style = "exist-ok"
experiment = "robot_controller_verify"
run = "verify_cnn_mlp_sample"

# If not None, set an external experiment path
expruns_path = None
# If not None, set an external results path
results_path = None

In [ ]:
if expruns_path:
    expruns_path = pathlib.Path(expruns_path).expanduser()
    assert expruns_path.exists()
    Config().set_exprun_path(expruns_path)
if results_path:
    results_path = pathlib.Path(results_path).expanduser()
    assert results_path.exists()
    Config().set_results_path(results_path)

exp = Config().get_experiment(experiment, run, creation_style=creation_style)
pprint(exp)

## Load the trained controller

In [ ]:
exp_trec = Config().get_experiment(exp["trec_experiment"], exp["trec_run"])
exp_roco = Config().get_experiment(
    exp_trec["controller"]["exp"], exp_trec["controller"]["run"])
exp_robot = Config().get_experiment(
    exp_trec["robot"]["exp"], exp_trec["robot"]["run"])
bundle_path = pathlib.Path(exp_trec.data_dir(), exp_trec["model_file"])
controller = create_controller(exp_roco, bundle_path=bundle_path)

# the sensor processing the encoder was trained with determines the preprocessing
exp_sp = controller.spec["components"][exp["sp_component"]]["sensor_exp"]
print(f"Controller bundle: {bundle_path}")

## Teacher-forced predictions on the testing demonstrations

In [ ]:
fields = RobotPosition.FIELDS
results = {}
for entry in exp["testing_data"]:
    demo = entry[1]
    predicted, target = teacher_forcing(
        controller, entry, exp_sp, exp_robot, len(fields),
        exp["input_label"], exp["output_label"])
    results[demo] = (predicted, target)
    np.savez(pathlib.Path(exp.data_dir(), f"predictions_{demo}.npz"),
             predicted=predicted, target=target)
    print(f"{demo}: {len(target)} predictions")

In [ ]:
# mean squared error of the normalized actions, per field
errors = {demo: ((predicted - target) ** 2).mean(axis=0)
          for demo, (predicted, target) in results.items()}
errors["all"] = np.concatenate(
    [(predicted - target) ** 2 for predicted, target in results.values()]).mean(axis=0)

errors_file = pathlib.Path(exp.data_dir(), "errors.csv")
with open(errors_file, "w") as f:
    f.write("demonstration," + ",".join(fields) + ",mean\n")
    for demo, mse in errors.items():
        f.write(f"{demo}," + ",".join(f"{value:.6f}" for value in mse)
                + f",{mse.mean():.6f}\n")
print(errors_file.read_text())

## Predicted versus recorded actions

In [ ]:
def save_figure(name):
    """Save the current figure as a PDF and a same-basename PNG preview."""
    for suffix in [".pdf", ".png"]:
        plt.savefig(pathlib.Path(exp.data_dir(), name + suffix), bbox_inches="tight")

for demo, (predicted, target) in results.items():
    fig, axes = plt.subplots(2, 3, figsize=(12, 6), sharex=True)
    for i, ax in enumerate(axes.flat):
        ax.plot(target[:, i], label="recorded")
        ax.plot(predicted[:, i], label="predicted")
        ax.set_title(fields[i])
        ax.set_ylim(-0.05, 1.05)
    axes.flat[0].legend()
    fig.suptitle(f"Teacher-forced predictions on {demo}")
    save_figure(f"prediction_{demo}")
    plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(fields, errors["all"])
ax.set_ylabel("MSE (normalized)")
ax.set_title(f"Per-field error on {len(results)} testing demonstrations")
save_figure("errors")
plt.show()

In [ ]:
exp.done()